In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 프리미티브 입력 및 출력

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** 새로운 실행 모델의 베타 버전이 제공됩니다. 지향형 실행 모델은 오류 완화 워크플로우를 맞춤 설정할 때 더 많은 유연성을 제공합니다. 자세한 내용은 [지향형 실행 모델](/guides/directed-execution-model) 가이드를 참조하세요.


<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 아래 요구 사항을 사용하여 개발되었습니다.
해당 버전 이상을 사용하시기 권장합니다.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

이 페이지는 IBM Quantum&reg; 컴퓨팅 리소스에서 워크로드를 실행하는 Qiskit Runtime 프리미티브의 입력 및 출력에 대한 개요를 제공합니다. 이 프리미티브는 **Primitive Unified Bloc(PUB)**이라는 데이터 구조를 사용하여 벡터화된 워크로드를 효율적으로 정의할 수 있는 기능을 제공합니다. 이러한 PUB는 QPU가 워크로드를 실행하기 위한 기본 작업 단위입니다. PUB는 Sampler 및 Estimator 프리미티브의 [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) 메서드에 입력으로 사용되며, 정의된 워크로드를 작업(job)으로 실행합니다. 그런 다음 작업이 완료되면 결과는 사용된 PUB와 Sampler 또는 Estimator 프리미티브에서 지정한 런타임 옵션 모두에 따라 달라지는 형식으로 반환됩니다.

<span id="pubs"></span>
## PUB 개요
프리미티브의 [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) 메서드를 호출할 때 필요한 주요 인수는 하나 이상의 튜플로 구성된 `list`입니다. 각 튜플은 프리미티브가 실행하는 Circuit에 해당합니다. 이 튜플 각각이 PUB이며, 목록의 각 튜플에 필요한 요소는 사용하는 프리미티브에 따라 다릅니다. 이 튜플에 제공되는 데이터는 브로드캐스팅을 통해 워크로드의 유연성을 제공하기 위해 다양한 형태로 배열될 수도 있습니다. 브로드캐스팅 규칙은 [이후 섹션](#broadcasting-rules)에서 설명합니다.

### Estimator PUB
Estimator 프리미티브의 경우 PUB 형식에는 최대 네 가지 값이 포함되어야 합니다.
- 하나 이상의 [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter) 객체를 포함할 수 있는 단일 `QuantumCircuit`
- 추정할 기댓값을 지정하는 하나 이상의 관측값(observable) 목록. 배열로 배치됩니다(예: 단일 관측값은 0차원 배열, 관측값 목록은 1차원 배열 등). 데이터는 `Pauli`, `SparsePauliOp`, `PauliList`, 또는 `str`과 같은 `ObservablesArrayLike` 형식 중 하나일 수 있습니다.
   > **Note:** 동일한 Circuit을 가진 서로 다른 PUB에 두 개의 가환(commuting) 관측값이 있는 경우, 동일한 측정을 사용하여 추정되지 않습니다. 각 PUB는 측정의 서로 다른 기저(basis)를 나타내므로, 각 PUB에 대해 별도의 측정이 필요합니다. 가환 관측값이 동일한 측정을 사용하여 추정되도록 하려면, 동일한 PUB 내에 그룹화해야 합니다.
- Circuit에 바인딩할 파라미터 값 컬렉션. 마지막 인덱스가 Circuit의 `Parameter` 객체에 대한 단일 배열형 객체로 지정하거나, Circuit에 `Parameter` 객체가 없는 경우 생략(또는 동등하게 `None`으로 설정)할 수 있습니다.
- (선택 사항) 추정할 기댓값의 목표 정밀도

### Sampler PUB
Sampler 프리미티브의 경우 PUB 튜플 형식에는 최대 세 가지 값이 포함됩니다.
- 하나 이상의 [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter) 객체를 포함할 수 있는 단일 `QuantumCircuit`
   *참고: 이 Circuit에는 샘플링할 각 Qubit에 대한 측정 명령도 포함되어야 합니다.*
- Circuit에 바인딩할 파라미터 값 컬렉션 $\theta_k$ (런타임에 바인딩해야 하는 `Parameter` 객체가 사용된 경우에만 필요)
- (선택 사항) Circuit을 측정할 샷(shot) 수
---

다음 코드는 `Estimator` 프리미티브에 대한 벡터화된 입력 예시를 보여주며, 이를 IBM&reg; Backend에서 단일 `RuntimeJobV2` 객체로 실행합니다.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray
from qiskit.primitives import StatevectorEstimator


import numpy as np

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit without providing a backend
pm = generate_preset_pass_manager(optimization_level=1)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 10),
        np.linspace(-4 * np.pi, 4 * np.pi, 10),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator = StatevectorEstimator()
estimator_pub = (transpiled_circuit, observables, params)

# Run the transpiled circuit
# using the set of parameters and observables.

job = estimator.run([estimator_pub])
result = job.result()

### 브로드캐스팅 규칙
PUB는 NumPy와 동일한 브로드캐스팅 규칙에 따라 여러 배열(관측값 및 파라미터 값)의 요소를 집계합니다. 이 섹션에서는 해당 규칙을 간략하게 요약합니다. 자세한 설명은 [NumPy 브로드캐스팅 규칙 문서](https://numpy.org/doc/stable/user/basics.broadcasting.html)를 참조하세요.

규칙:

* 입력 배열의 차원 수가 동일할 필요는 없습니다.
  * 결과 배열은 가장 큰 차원을 가진 입력 배열과 동일한 차원 수를 갖습니다.
  * 각 차원의 크기는 해당 차원의 가장 큰 크기입니다.
  * 누락된 차원은 크기가 1인 것으로 가정합니다.
* 형태 비교는 가장 오른쪽 차원에서 시작하여 왼쪽으로 진행합니다.
* 두 차원의 크기가 같거나 그 중 하나가 1이면 호환됩니다.

브로드캐스팅되는 배열 쌍의 예:

In [2]:
# Broadcast single observable
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = SparsePauliOp("ZZZ")  # shape ()
# >> pub result has shape (5,)

# Zip
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = [
    SparsePauliOp(pauli) for pauli in ["III", "XXX", "YYY", "ZZZ", "XYZ"]
]  # shape (5,)
# >> pub result has shape (5,)

# Outer/Product
parameter_values = np.random.uniform(size=(1, 6))  # shape (1, 6)
observables = [
    [SparsePauliOp(pauli)] for pauli in ["III", "XXX", "YYY", "ZZZ"]
]  # shape (4, 1)
# >> pub result has shape (4, 6)

# Standard nd generalization
parameter_values = np.random.uniform(size=(3, 6))  # shape (3, 6)
observables = [
    [
        [SparsePauliOp(["XII"])],
        [SparsePauliOp(["IXI"])],
        [SparsePauliOp(["IIX"])],
    ],
    [
        [SparsePauliOp(["ZII"])],
        [SparsePauliOp(["IZI"])],
        [SparsePauliOp(["IIZ"])],
    ],
]  # shape (2, 3, 1)
# >> pub result has shape (2, 3, 6)

브로드캐스팅되지 않는 배열 쌍의 예:

In [3]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 10), dtype=float64>), stds=np.ndarray(<shape=(3, 10), dtype=float64>), shape=(3, 10)), metadata={'target_precision': 0.0, 'circuit_metadata': {}})], metadata={'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 10), dtype=float64>), stds=np.ndarray(<shape=(3, 10), dtype=float64>), shape=(3, 10))

And this DataBin has attributes: dict_keys(['evs', 'stds'])
Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). 

The expectation values measured from this PUB are: 
[[ 3.06161700e-16  4.52395120e-01  4.36594428e-01  2.16506351e-01
   6.33718361e-01 -6.33718361e-01 -2.16506351e-01 -4.36594428e-01
  -4.52395120e-01 -3.06161700e-16]
 [ 1.22464680

`EstimatorV2`는 브로드캐스팅된 형태의 각 요소에 대해 하나의 기댓값 추정치를 반환합니다.

다음은 배열 브로드캐스팅으로 표현된 일반적인 패턴의 몇 가지 예입니다. 아래 그림에서 이에 대한 시각적 표현을 확인할 수 있습니다.

파라미터 값 세트는 n x m 배열로, 관측값 배열은 하나 이상의 단일 열 배열로 표현됩니다. 이전 코드의 각 예시에서 파라미터 값 세트는 관측값 배열과 결합하여 결과 기댓값 추정치를 생성합니다.

 - *예시 1*: (단일 관측값 브로드캐스팅) 파라미터 값 세트는 5x1 배열이고 관측값 배열은 1x1 배열입니다. 관측값 배열의 항목 하나가 파라미터 값 세트의 각 항목과 결합되어, 원래 파라미터 값 세트의 항목과 관측값 배열의 항목의 조합으로 이루어진 단일 5x1 배열이 생성됩니다.

 - *예시 2*: (zip) 5x1 파라미터 값 세트와 5x1 관측값 배열이 있습니다. 출력은 파라미터 값 세트의 n번째 항목과 관측값 배열의 n번째 항목이 결합된 5x1 배열입니다.

  - *예시 3*: (외적/곱) 1x6 파라미터 값 세트와 4x1 관측값 배열이 있습니다. 이들의 조합은 파라미터 값 세트의 각 항목이 관측값 배열의 *모든* 항목과 결합되어 생성된 4x6 배열이며, 따라서 각 파라미터 값은 출력에서 전체 열이 됩니다.

  - *예시 4*: (표준 n차원 일반화) 3x6 파라미터 값 세트 배열과 두 개의 3x1 관측값 배열이 있습니다. 이들은 이전 예시와 유사한 방식으로 결합하여 두 개의 3x6 출력 배열을 만들어냅니다.

![이 이미지는 배열 브로드캐스팅의 여러 시각적 표현을 보여줍니다](../docs/images/guides/primitive-input-output/broadcasting.svg "브로드캐스팅의 시각적 표현")

In [4]:
from qiskit.primitives import StatevectorSampler

# generate a ten-qubit GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure_all()

# transpile the circuit
transpiled_circuit = pm.run(circuit)

sampler = StatevectorSampler()

# run the Sampler job and retrieve the results

job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains one BitArray
data = result[0].data
print(f"Databin: {data}\n")

# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method
array = data.meas
print(f"BitArray: {array}\n")
print(f"The shape of register `meas` is {data.meas.array.shape}.\n")
print(f"The bytes in register `alpha`, shot by shot:\n{data.meas.array}\n")

Databin: DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=10>))

BitArray: BitArray(<shape=(), num_shots=1024, num_bits=10>)

The shape of register `meas` is (1024, 2).

The bytes in register `alpha`, shot by shot:
[[  0   0]
 [  3 255]
 [  0   0]
 ...
 [  3 255]
 [  3 255]
 [  3 255]]



<Admonition type="tip" title="SparsePauliOp">
각 `SparsePauliOp`는 `SparsePauliOp`에 포함된 Pauli의 수에 관계없이 이 맥락에서 단일 요소로 계산됩니다. 따라서 이 브로드캐스팅 규칙의 목적상 다음 요소들은 모두 동일한 형태를 가집니다.

In [5]:
# optionally, convert away from the native BitArray format to a dictionary format
counts = data.meas.get_counts()
print(f"Counts: {counts}")

Counts: {'0000000000': 492, '1111111111': 532}


다음 연산자 목록들은 포함된 정보 측면에서 동등하지만 형태가 다릅니다.

In [6]:
# generate a ten-qubit GHZ circuit with two classical registers
circuit = QuantumCircuit(
    qreg := QuantumRegister(10),
    alpha := ClassicalRegister(1, "alpha"),
    beta := ClassicalRegister(9, "beta"),
)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure([0], alpha)
circuit.measure(range(1, 10), beta)

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results

job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains two BitArrays, one per register, and can be accessed
# as attributes using the registers' names
data = result[0].data
print(f"BitArray for register 'alpha': {data.alpha}")
print(f"BitArray for register 'beta': {data.beta}")

BitArray for register 'alpha': BitArray(<shape=(), num_shots=1024, num_bits=1>)
BitArray for register 'beta': BitArray(<shape=(), num_shots=1024, num_bits=9>)


</Admonition>
## 프리미티브 출력 개요
하나 이상의 PUB이 실행을 위해 QPU에 전송되고 작업이 성공적으로 완료되면, 데이터는 `RuntimeJobV2.result()` 메서드를 호출하여 접근하는 [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) 컨테이너 객체로 반환됩니다. `PrimitiveResult`에는 각 PUB의 실행 결과를 담은 [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) 객체의 이터러블 목록이 포함됩니다. 사용된 프리미티브에 따라, 이 데이터는 Estimator의 경우 기댓값과 오차 막대이거나, Sampler의 경우 Circuit 출력의 샘플입니다.

이 목록의 각 요소는 프리미티브의 `run()` 메서드에 제출된 각 PUB에 해당합니다(예: 20개의 PUB으로 제출된 작업은 각 PUB에 해당하는 20개의 `PubResult` 목록을 포함하는 `PrimitiveResult` 객체를 반환합니다).

각 `PubResult` 객체는 `data` 속성과 `metadata` 속성을 모두 가집니다. `data` 속성은 실제 측정값, 표준 편차 등을 포함하는 사용자 정의 [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin)입니다. 이 `DataBin`은 연관된 PUB의 형태나 구조, 그리고 작업 제출에 사용된 프리미티브가 지정한 오류 완화 옵션(예: [ZNE](./error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) 또는 [PEC](./error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec))에 따라 다양한 속성을 가집니다. 한편 `metadata` 속성에는 런타임 및 오류 완화 옵션에 관한 정보가 포함됩니다(이 페이지의 [결과 메타데이터](#result-metadata) 섹션에서 나중에 설명됩니다).

다음은 `PrimitiveResult` 데이터 구조를 시각적으로 나타낸 개요입니다:

<Tabs>
    <TabItem value="estimator" label="Estimator Output">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
    <TabItem value="sampler" label="Sampler Output">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── NAME_OF_CLASSICAL_REGISTER
        │       │   └── BitArray of count data (default is 'meas')
        |       |
        │       └── NAME_OF_ANOTHER_CLASSICAL_REGISTER
        │           └── BitArray of count data (exists only if more than one
        |                 ClassicalRegister was specified in the circuit)
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       └── NAME_OF_CLASSICAL_REGISTER
        |           └── BitArray of count data for second pub
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
</Tabs>

간단히 말하면, 단일 작업은 `PrimitiveResult` 객체를 반환하며 하나 이상의 `PubResult` 객체 목록을 포함합니다. 이 `PubResult` 객체들은 작업에 제출된 각 PUB의 측정 데이터를 저장합니다.

각 `PubResult`는 작업에 사용된 프리미티브 유형에 따라 서로 다른 형식과 속성을 가집니다. 자세한 내용은 아래에 설명되어 있습니다.

### Estimator 출력
Estimator 프리미티브의 각 `PubResult`에는 최소한 기댓값 배열(`PubResult.data.evs`)과 관련 표준 편차(사용된 `resilience_level`에 따라 `PubResult.data.stds` 또는 `PubResult.data.ensemble_standard_error`)가 포함되며, 지정된 오류 완화 옵션에 따라 더 많은 데이터가 포함될 수 있습니다.

아래 코드 스니펫은 위에서 생성된 작업에 대한 `PrimitiveResult`(및 관련 `PubResult`) 형식을 설명합니다.

In [7]:
print(f"The shape of register `alpha` is {data.alpha.array.shape}.")
print(f"The bytes in register `alpha`, shot by shot:\n{data.alpha.array}\n")

print(f"The shape of register `beta` is {data.beta.array.shape}.")
print(f"The bytes in register `beta`, shot by shot:\n{data.beta.array}\n")

# post-select the bitstrings of `beta` based on having sampled "1" in `alpha`
mask = data.alpha.array == "0b1"
ps_beta = data.beta[mask[:, 0]]
print(f"The shape of `beta` after post-selection is {ps_beta.array.shape}.")
print(f"The bytes in `beta` after post-selection:\n{ps_beta.array}")

# get a slice of `beta` to retrieve the first three bits
beta_sl_bits = data.beta.slice_bits([0, 1, 2])
print(
    f"The shape of `beta` after bit-wise slicing is {beta_sl_bits.array.shape}."
)
print(f"The bytes in `beta` after bit-wise slicing:\n{beta_sl_bits.array}\n")

# get a slice of `beta` to retrieve the bytes of the first five shots
beta_sl_shots = data.beta.slice_shots([0, 1, 2, 3, 4])
print(
    f"The shape of `beta` after shot-wise slicing is {beta_sl_shots.array.shape}."
)
print(
    f"The bytes in `beta` after shot-wise slicing:\n{beta_sl_shots.array}\n"
)

# calculate the expectation value of diagonal operators on `beta`
ops = [SparsePauliOp("ZZZZZZZZZ"), SparsePauliOp("IIIIIIIIZ")]
exp_vals = data.beta.expectation_values(ops)
for o, e in zip(ops, exp_vals):
    print(f"Exp. val. for observable `{o}` is: {e}")

# concatenate the bitstrings in `alpha` and `beta` to "merge" the results of the two
# registers
merged_results = BitArray.concatenate_bits([data.alpha, data.beta])
print(f"\nThe shape of the merged results is {merged_results.array.shape}.")
print(f"The bytes of the merged results:\n{merged_results.array}\n")

The shape of register `alpha` is (1024, 1).
The bytes in register `alpha`, shot by shot:
[[1]
 [1]
 [1]
 ...
 [0]
 [0]
 [1]]

The shape of register `beta` is (1024, 2).
The bytes in register `beta`, shot by shot:
[[  1 255]
 [  1 255]
 [  1 255]
 ...
 [  0   0]
 [  0   0]
 [  1 255]]

The shape of `beta` after post-selection is (0, 2).
The bytes in `beta` after post-selection:
[]
The shape of `beta` after bit-wise slicing is (1024, 1).
The bytes in `beta` after bit-wise slicing:
[[7]
 [7]
 [7]
 ...
 [0]
 [0]
 [7]]

The shape of `beta` after shot-wise slicing is (5, 2).
The bytes in `beta` after shot-wise slicing:
[[  1 255]
 [  1 255]
 [  1 255]
 [  1 255]
 [  1 255]]



Exp. val. for observable `SparsePauliOp(['ZZZZZZZZZ'],
              coeffs=[1.+0.j])` is: -0.017578125
Exp. val. for observable `SparsePauliOp(['IIIIIIIIZ'],
              coeffs=[1.+0.j])` is: -0.017578125

The shape of the merged results is (1024, 2).
The bytes of the merged results:
[[  3 255]
 [  3 255]
 [  3 255]
 ...
 [  0   0]
 [  0   0]
 [  3 255]]



## Result metadata

In addition to the execution results, the `PrimitiveResult` and `PubResult` objects contain an optional metadata attribute about the job that was submitted. The metadata returned (if any) is implementation-specific.

In [8]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'version' : 2,

The metadata of the PubResult result is:
'shots' : 1024,
'circuit_metadata' : {},


#### Estimator가 오류를 계산하는 방법
입력 PUB에 전달된 관측 가능량의 평균 추정값(`DataBin`의 `evs` 필드) 외에도, Estimator는 해당 기댓값과 관련된 오류의 추정값도 제공하려 합니다. 모든 Estimator 쿼리는 각 기댓값에 대한 평균의 표준 오차와 유사한 양으로 `stds` 필드를 채우지만, 일부 오류 완화 옵션은 `ensemble_standard_error`와 같은 추가 정보를 생성합니다.

단일 관측 가능량 $\mathcal{O}$를 고려해 봅시다. [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne)가 없는 경우, Estimator 실행의 각 샷은 기댓값 $\langle \mathcal{O} \rangle$의 점 추정값을 제공한다고 생각할 수 있습니다. 점별 추정값이 벡터 `Os`에 있다면, `ensemble_standard_error`에 반환되는 값은 다음과 동일합니다($\sigma_{\mathcal{O}}$는 [기댓값의 표준 편차](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) 추정값이고 $N_{shots}$는 샷 수입니다):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

이는 모든 샷을 단일 앙상블의 일부로 취급합니다. Gate [twirling](/guides/error-mitigation-and-suppression-techniques#pauli-twirling)(`twirling.enable_gates = True`)을 요청한 경우, $\langle \mathcal{O} \rangle$의 점별 추정값을 공통 트윌을 공유하는 집합으로 정렬할 수 있습니다. 이 추정값 집합을 `O_twirls`라고 부르며, 이러한 집합이 `num_randomizations`(트윌 수)만큼 있습니다. 그러면 `stds`는 다음과 같이 `O_twirls`의 평균의 표준 오차입니다:

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

여기서 $\sigma_{\mathcal{O}}$는 `O_twirls`의 표준 편차이고 $N_{twirls}$는 트윌 수입니다. twirling을 활성화하지 않으면 `stds`와 `ensemble_standard_error`는 동일합니다.

ZNE를 활성화하면, 위에서 설명한 `stds`는 외삽기 모델에 대한 비선형 회귀의 가중치가 됩니다. 이 경우 `stds`에 최종적으로 반환되는 것은 잡음 인자가 0일 때 평가된 적합 모델의 불확실성입니다. 적합이 불량하거나 적합의 불확실성이 클 경우, 보고된 `stds`가 매우 커질 수 있습니다. ZNE가 활성화되면 `pub_result.data.evs_noise_factors`와 `pub_result.data.stds_noise_factors`도 채워지므로, 직접 외삽을 수행할 수 있습니다.
### Sampler 출력
Sampler 작업이 성공적으로 완료되면, 반환되는 [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) 객체에는 PUB별로 하나씩 [`SamplerPubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.SamplerPubResult) 목록이 포함됩니다. 이 `SamplerPubResult` 객체의 데이터 빈(data bin)은 Circuit의 각 `ClassicalRegister`마다 하나의 `BitArray`를 포함하는 딕셔너리 유사 객체입니다.

`BitArray` 클래스는 순서가 있는 샷(shot) 데이터를 담는 컨테이너입니다. 더 자세히 설명하면, 샘플링된 비트 문자열을 2차원 배열 내에 바이트 형태로 저장합니다. 이 배열의 가장 왼쪽 축은 순서가 있는 샷에 걸쳐 있으며, 가장 오른쪽 축은 바이트에 걸쳐 있습니다.

첫 번째 예시로, 다음 10-Qubit Circuit을 살펴보겠습니다.